# Code

In [1]:
# ============================================================
# Fast MABe Extra Trees Pipeline (Parallelized)
# Combines the high-scoring pipeline of Extra Trees with 
# the parallel execution structure of the Improved notebook.
# ============================================================

import os
import sys
import gc
import re
import ast
import json
import math
import shutil
import random
import itertools
import warnings
import multiprocessing
from pathlib import Path
from collections import defaultdict
from time import perf_counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from scipy import signal, stats
from scipy.ndimage import median_filter

# ML Libraries
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.base import ClassifierMixin, BaseEstimator, clone
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline

warnings.filterwarnings('ignore')

# ============================================================
# Configuration
# ============================================================
# Local paths
DATA_DIR = "../" # Assuming data is in parent dir relative to notebook location in kaggle_comp
# Adjust if necessary based on your workspace structure
if not os.path.exists(os.path.join(DATA_DIR, 'train.csv')):
    # Fallback to absolute path if relative fails
    DATA_DIR = "/kaggle/input/MABe-mouse-behavior-detection"

INPUT_DIR = Path(DATA_DIR)
TRAIN_TRACKING_DIR = INPUT_DIR / "train_tracking"
TRAIN_ANNOTATION_DIR = INPUT_DIR / "train_annotation"
TEST_TRACKING_DIR = INPUT_DIR / "test_tracking"

WORKING_DIR = Path("results_fast_extra_trees")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# GPU Check
USE_GPU = shutil.which("nvidia-smi") is not None
print(f'Using GPU? {USE_GPU}')

SEED = 1234
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ============================================================
# Model Definitions
# ============================================================

def _make_lgbm(**kw):
    kw.setdefault("random_state", SEED)
    kw.setdefault("feature_fraction_seed", SEED)
    kw.setdefault("data_random_seed", SEED)
    # Force CPU for LightGBM to avoid "Check failed: (best_split_info.left_count) > (0)" error on GPU
    # LightGBM is highly optimized for CPU and this prevents the fatal crash on small/imbalanced subsets.
    kw.setdefault("device", 'cpu')
    return lgb.LGBMClassifier(**kw)

def _make_xgb(**kw):
    kw.setdefault("random_state", SEED)
    kw.setdefault("tree_method", "gpu_hist" if USE_GPU else "hist")
    return xgb.XGBClassifier(**kw)

def _make_cb(**kw):
    kw.setdefault("random_seed", SEED)
    if USE_GPU:
        kw.setdefault("task_type", "GPU")
        kw.setdefault("devices", "0")
    else:
        kw.setdefault("task_type", "CPU")
    return CatBoostClassifier(**kw)

class StratifiedSubsetClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator, n_samples, random_state=SEED):
        self.estimator = estimator
        self.n_samples = n_samples and int(n_samples)
        self.random_state = random_state

    def fit(self, X, y):
        y = np.asarray(y)
        n_total = len(y)

        if self.n_samples is None or self.n_samples >= n_total:
            rng = np.random.default_rng(self.random_state)
            idx = rng.permutation(n_total)
        else:
            sss = StratifiedShuffleSplit(
                n_splits=1, train_size=self.n_samples, random_state=self.random_state
            )
            idx, _ = next(sss.split(np.zeros(n_total, dtype=np.int8), y))

        Xn = X.iloc[idx]
        Xn = Xn.to_numpy(np.float32, copy=False)
        yn = y[idx]

        self.estimator.fit(Xn, yn)
        self.classes_ = getattr(self.estimator, "classes_", np.array([0, 1]))
        return self

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

    def predict(self, X):
        return self.estimator.predict(X)

class StratifiedSubsetClassifierWEval(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator, n_samples=None, random_state=42, valid_size=0.10, val_cap_ratio=0.25, es_rounds="auto", es_metric="auto"):
        self.estimator = estimator
        self.n_samples = (int(n_samples) if (n_samples is not None) else None)
        self.random_state = random_state
        self.valid_size = float(valid_size)
        self.val_cap_ratio = float(val_cap_ratio)
        self.es_rounds = es_rounds
        self.es_metric = es_metric
 
    def fit(self, X: pd.DataFrame, y):
        y = np.asarray(y)
        n_total = len(y)
        tr_idx, va_idx = self._compute_train_val_indices(y, n_total)
        Xtr = X.iloc[tr_idx]; ytr = y[tr_idx]
        Xtr = Xtr.to_numpy(np.float32, copy=False)

        Xva = yva = None
        if va_idx is not None and len(va_idx) > 0:
            Xva = X.iloc[va_idx].to_numpy(np.float32, copy=False); yva = y[va_idx]

        # Fit with ES if we have any validation
        if (Xva is not None and len(yva) > 0):
            if isinstance(self.estimator, xgb.XGBClassifier):
                self.estimator.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False, early_stopping_rounds=50)
            elif isinstance(self.estimator, CatBoostClassifier):
                self.estimator.fit(Xtr, ytr, eval_set=(Xva, yva), verbose=False, early_stopping_rounds=50)
            else:
                self.estimator.fit(Xtr, ytr)
        else:
            self.estimator.fit(Xtr, ytr)

        self.classes_ = getattr(self.estimator, "classes_", np.array([0, 1]))
        return self

    def predict_proba(self, X: pd.DataFrame):
        return self.estimator.predict_proba(X)

    def predict(self, X: pd.DataFrame):
        return self.estimator.predict(X)

    def _compute_train_val_indices(self, y: np.ndarray, n_total: int):
        rng = np.random.default_rng(self.random_state)
        if self.n_samples is None or self.n_samples >= n_total:
             idx = rng.permutation(n_total); return idx, None
        
        sss_tr = StratifiedShuffleSplit(n_splits=1, train_size=self.n_samples, random_state=self.random_state)
        tr_idx, rest_idx = next(sss_tr.split(np.zeros(n_total, dtype=np.int8), y))
        
        # Simple validation split from rest
        if len(rest_idx) > 100:
             va_idx = rest_idx[:int(len(rest_idx)*0.5)] # Take half of rest as valid
        else:
             va_idx = None
        return tr_idx, va_idx

# ============================================================
# Feature Engineering Helpers (From Extra Trees Notebook)
# ============================================================

def _scale(n_frames_at_30fps, fps, ref=30.0):
    return max(1, int(round(n_frames_at_30fps * float(fps) / ref)))

def _scale_signed(n_frames_at_30fps, fps, ref=30.0):
    if n_frames_at_30fps == 0: return 0
    s = 1 if n_frames_at_30fps > 0 else -1
    mag = max(1, int(round(abs(n_frames_at_30fps) * float(fps) / ref)))
    return s * mag

def _speed(cx: pd.Series, cy: pd.Series, fps: float) -> pd.Series:
    return np.hypot(cx.diff(), cy.diff()).fillna(0.0) * float(fps)

def _roll_future_mean(s: pd.Series, w: int, min_p: int = 1) -> pd.Series:
    return s.iloc[::-1].rolling(w, min_periods=min_p).mean().iloc[::-1]

def _roll_future_var(s: pd.Series, w: int, min_p: int = 2) -> pd.Series:
    return s.iloc[::-1].rolling(w, min_periods=min_p).var().iloc[::-1]

def add_curvature_features(X, center_x, center_y, fps):
    vel_x = center_x.diff(); vel_y = center_y.diff()
    acc_x = vel_x.diff(); acc_y = vel_y.diff()
    cross_prod = vel_x * acc_y - vel_y * acc_x
    vel_mag = np.sqrt(vel_x**2 + vel_y**2)
    curvature = np.abs(cross_prod) / (vel_mag**3 + 1e-6)
    for w in [30, 60]:
        ws = _scale(w, fps)
        X[f'curv_mean_{w}'] = curvature.rolling(ws, min_periods=max(1, ws // 6)).mean()
    angle = np.arctan2(vel_y, vel_x)
    angle_change = np.abs(angle.diff())
    w = 30; ws = _scale(w, fps)
    X[f'turn_rate_{w}'] = angle_change.rolling(ws, min_periods=max(1, ws // 6)).sum()
    return X

def add_multiscale_features(X, center_x, center_y, fps):
    speed = np.sqrt(center_x.diff()**2 + center_y.diff()**2) * float(fps)
    scales = [10, 40, 160]
    for scale in scales:
        ws = _scale(scale, fps)
        if len(speed) >= ws:
            X[f'sp_m{scale}'] = speed.rolling(ws, min_periods=max(1, ws // 4)).mean()
            X[f'sp_s{scale}'] = speed.rolling(ws, min_periods=max(1, ws // 4)).std()
    if len(scales) >= 2 and f'sp_m{scales[0]}' in X.columns and f'sp_m{scales[-1]}' in X.columns:
        X['sp_ratio'] = X[f'sp_m{scales[0]}'] / (X[f'sp_m{scales[-1]}'] + 1e-6)
    return X

def add_state_features(X, center_x, center_y, fps):
    speed = np.sqrt(center_x.diff()**2 + center_y.diff()**2) * float(fps)
    w_ma = _scale(15, fps)
    speed_ma = speed.rolling(w_ma, min_periods=max(1, w_ma // 3)).mean()
    try:
        bins = [-np.inf, 0.5 * fps, 2.0 * fps, 5.0 * fps, np.inf]
        speed_states = pd.cut(speed_ma, bins=bins, labels=[0, 1, 2, 3]).astype(float)
        for window in [60, 120]:
            ws = _scale(window, fps)
            if len(speed_states) >= ws:
                for state in [0, 1, 2, 3]:
                    X[f's{state}_{window}'] = ((speed_states == state).astype(float).rolling(ws, min_periods=max(1, ws // 6)).mean())
                state_changes = (speed_states != speed_states.shift(1)).astype(float)
                X[f'trans_{window}'] = state_changes.rolling(ws, min_periods=max(1, ws // 6)).sum()
    except Exception: pass
    return X

def add_longrange_features(X, center_x, center_y, fps):
    for window in [120, 240]:
        ws = _scale(window, fps)
        if len(center_x) >= ws:
            X[f'x_ml{window}'] = center_x.rolling(ws, min_periods=max(5, ws // 6)).mean()
            X[f'y_ml{window}'] = center_y.rolling(ws, min_periods=max(5, ws // 6)).mean()
    for span in [60, 120]:
        s = _scale(span, fps)
        X[f'x_e{span}'] = center_x.ewm(span=s, min_periods=1).mean()
        X[f'y_e{span}'] = center_y.ewm(span=s, min_periods=1).mean()
    speed = np.sqrt(center_x.diff()**2 + center_y.diff()**2) * float(fps)
    for window in [60, 120]:
        ws = _scale(window, fps)
        if len(speed) >= ws:
            X[f'sp_pct{window}'] = speed.rolling(ws, min_periods=max(5, ws // 6)).rank(pct=True)
    return X

def add_cumulative_distance_single(X, cx, cy, fps, horizon_frames_base=180, colname="path_cum180"):
    L = max(1, _scale(horizon_frames_base, fps))
    step = np.hypot(cx.diff(), cy.diff())
    path = step.rolling(2*L + 1, min_periods=max(5, L//6), center=True).sum()
    X[colname] = path.fillna(0.0).astype(np.float32)
    return X

def add_groom_microfeatures(X, df, fps):
    parts = df.columns.get_level_values(0)
    if 'body_center' not in parts or 'nose' not in parts: return X
    cx = df['body_center']['x']; cy = df['body_center']['y']
    nx = df['nose']['x']; ny = df['nose']['y']
    cs = (np.sqrt(cx.diff()**2 + cy.diff()**2) * float(fps)).fillna(0)
    ns = (np.sqrt(nx.diff()**2 + ny.diff()**2) * float(fps)).fillna(0)
    w30 = _scale(30, fps)
    X['head_body_decouple'] = (ns / (cs + 1e-3)).clip(0, 10).rolling(w30, min_periods=max(1, w30//3)).median()
    r = np.sqrt((nx - cx)**2 + (ny - cy)**2)
    X['nose_rad_std'] = r.rolling(w30, min_periods=max(1, w30//3)).std().fillna(0)
    if 'tail_base' in parts:
        ang = np.arctan2(df['nose']['y']-df['tail_base']['y'], df['nose']['x']-df['tail_base']['x'])
        dang = np.abs(ang.diff()).fillna(0)
        X['head_orient_jitter'] = dang.rolling(w30, min_periods=max(1, w30//3)).mean()
    return X

def add_interaction_features(X, mouse_pair, avail_A, avail_B, fps):
    if 'body_center' not in avail_A or 'body_center' not in avail_B: return X
    rel_x = mouse_pair['A']['body_center']['x'] - mouse_pair['B']['body_center']['x']
    rel_y = mouse_pair['A']['body_center']['y'] - mouse_pair['B']['body_center']['y']
    rel_dist = np.sqrt(rel_x**2 + rel_y**2)
    A_vx = mouse_pair['A']['body_center']['x'].diff(); A_vy = mouse_pair['A']['body_center']['y'].diff()
    B_vx = mouse_pair['B']['body_center']['x'].diff(); B_vy = mouse_pair['B']['body_center']['y'].diff()
    A_lead = (A_vx * rel_x + A_vy * rel_y) / (np.sqrt(A_vx**2 + A_vy**2) * rel_dist + 1e-6)
    B_lead = (B_vx * (-rel_x) + B_vy * (-rel_y)) / (np.sqrt(B_vx**2 + B_vy**2) * rel_dist + 1e-6)
    for window in [30, 60]:
        ws = _scale(window, fps)
        X[f'A_ld{window}'] = A_lead.rolling(ws, min_periods=max(1, ws // 6)).mean()
        X[f'B_ld{window}'] = B_lead.rolling(ws, min_periods=max(1, ws // 6)).mean()
    approach = -rel_dist.diff()
    chase = approach * B_lead
    w = 30; ws = _scale(w, fps)
    X[f'chase_{w}'] = chase.rolling(ws, min_periods=max(1, ws // 6)).mean()
    for window in [60, 120]:
        ws = _scale(window, fps)
        A_sp = np.sqrt(A_vx**2 + A_vy**2); B_sp = np.sqrt(B_vx**2 + B_vy**2)
        X[f'sp_cor{window}'] = A_sp.rolling(ws, min_periods=max(1, ws // 6)).corr(B_sp)
    return X

def add_speed_asymmetry_future_past_single(X, cx, cy, fps, horizon_base=30, agg="mean"):
    w = max(3, _scale(horizon_base, fps))
    v = _speed(cx, cy, fps)
    if agg == "median":
        v_past = v.rolling(w, min_periods=max(3, w//4), center=False).median()
        v_fut  = v.iloc[::-1].rolling(w, min_periods=max(3, w//4)).median().iloc[::-1]
    else:
        v_past = v.rolling(w, min_periods=max(3, w//4), center=False).mean()
        v_fut  = _roll_future_mean(v, w, min_p=max(3, w//4))
    X["spd_asym_1s"] = (v_fut - v_past).fillna(0.0)
    return X

def add_gauss_shift_speed_future_past_single(X, cx, cy, fps, window_base=30, eps=1e-6):
    w = max(5, _scale(window_base, fps))
    v = _speed(cx, cy, fps)
    mu_p = v.rolling(w, min_periods=max(3, w//4)).mean()
    va_p = np.maximum(v.rolling(w, min_periods=max(3, w//4)).var(), eps)
    mu_f = _roll_future_mean(v, w, min_p=max(3, w//4))
    va_f = np.maximum(_roll_future_var(v, w, min_p=max(3, w//4)), eps)
    kl_pf = 0.5 * ((va_p/va_f) + ((mu_f - mu_p)**2)/va_f - 1.0 + np.log(va_f/va_p))
    kl_fp = 0.5 * ((va_f/va_p) + ((mu_p - mu_f)**2)/va_p - 1.0 + np.log(va_p/va_f))
    X["spd_symkl_1s"] = (kl_pf + kl_fp).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X

def transform_single(single_mouse, body_parts_tracked, fps):
    available_body_parts = single_mouse.columns.get_level_values(0)
    X = pd.DataFrame({
        f"{p1}+{p2}": np.square(single_mouse[p1] - single_mouse[p2]).sum(axis=1, skipna=False)
        for p1, p2 in itertools.combinations(body_parts_tracked, 2)
        if p1 in available_body_parts and p2 in available_body_parts
    })
    X = X.reindex(columns=[f"{p1}+{p2}" for p1, p2 in itertools.combinations(body_parts_tracked, 2)], copy=False)

    if all(p in single_mouse.columns for p in ['ear_left', 'ear_right', 'tail_base']):
        lag = _scale(10, fps)
        shifted = single_mouse[['ear_left', 'ear_right', 'tail_base']].shift(lag)
        speeds = pd.DataFrame({
            'sp_lf': np.square(single_mouse['ear_left'] - shifted['ear_left']).sum(axis=1, skipna=False),
            'sp_rt': np.square(single_mouse['ear_right'] - shifted['ear_right']).sum(axis=1, skipna=False),
            'sp_lf2': np.square(single_mouse['ear_left'] - shifted['tail_base']).sum(axis=1, skipna=False),
            'sp_rt2': np.square(single_mouse['ear_right'] - shifted['tail_base']).sum(axis=1, skipna=False),
        })
        X = pd.concat([X, speeds], axis=1)

    if 'nose+tail_base' in X.columns and 'ear_left+ear_right' in X.columns:
        X['elong'] = X['nose+tail_base'] / (X['ear_left+ear_right'] + 1e-6)

    if all(p in available_body_parts for p in ['nose', 'body_center', 'tail_base']):
        v1 = single_mouse['nose'] - single_mouse['body_center']
        v2 = single_mouse['tail_base'] - single_mouse['body_center']
        X['body_ang'] = (v1['x'] * v2['x'] + v1['y'] * v2['y']) / (
            np.sqrt(v1['x']**2 + v1['y']**2) * np.sqrt(v2['x']**2 + v2['y']**2) + 1e-6)

    if 'body_center' in available_body_parts:
        cx = single_mouse['body_center']['x']; cy = single_mouse['body_center']['y']
        for w in [5, 15, 30, 60]:
            ws = _scale(w, fps)
            roll = dict(min_periods=1, center=True)
            X[f'cx_m{w}'] = cx.rolling(ws, **roll).mean()
            X[f'cy_m{w}'] = cy.rolling(ws, **roll).mean()
            X[f'cx_s{w}'] = cx.rolling(ws, **roll).std()
            X[f'cy_s{w}'] = cy.rolling(ws, **roll).std()
            X[f'x_rng{w}'] = cx.rolling(ws, **roll).max() - cx.rolling(ws, **roll).min()
            X[f'y_rng{w}'] = cy.rolling(ws, **roll).max() - cy.rolling(ws, **roll).min()
            X[f'disp{w}'] = np.sqrt(cx.diff().rolling(ws, min_periods=1).sum()**2 + cy.diff().rolling(ws, min_periods=1).sum()**2)
            X[f'act{w}'] = np.sqrt(np.maximum(cx.diff().rolling(ws, min_periods=1).var() + cy.diff().rolling(ws, min_periods=1).var(), 0))

        X = add_curvature_features(X, cx, cy, fps)
        X = add_multiscale_features(X, cx, cy, fps)
        X = add_state_features(X, cx, cy, fps)
        X = add_longrange_features(X, cx, cy, fps)
        X = add_cumulative_distance_single(X, cx, cy, fps, horizon_frames_base=180)
        X = add_groom_microfeatures(X, single_mouse, fps)
        X = add_speed_asymmetry_future_past_single(X, cx, cy, fps, horizon_base=30)         
        X = add_gauss_shift_speed_future_past_single(X, cx, cy, fps, window_base=30)
  
    if all(p in available_body_parts for p in ['nose', 'tail_base']):
        nt_dist = np.sqrt((single_mouse['nose']['x'] - single_mouse['tail_base']['x'])**2 + (single_mouse['nose']['y'] - single_mouse['tail_base']['y'])**2)
        for lag in [10, 20, 40]:
            l = _scale(lag, fps)
            X[f'nt_lg{lag}'] = nt_dist.shift(l)
            X[f'nt_df{lag}'] = nt_dist - nt_dist.shift(l)

    if all(p in available_body_parts for p in ['ear_left', 'ear_right']):
        ear_d = np.sqrt((single_mouse['ear_left']['x'] - single_mouse['ear_right']['x'])**2 + (single_mouse['ear_left']['y'] - single_mouse['ear_right']['y'])**2)
        for off in [-20, -10, 10, 20]:
            o = _scale_signed(off, fps)
            X[f'ear_o{off}'] = ear_d.shift(-o)  
        w = _scale(30, fps)
        X['ear_con'] = ear_d.rolling(w, min_periods=1, center=True).std() / (ear_d.rolling(w, min_periods=1, center=True).mean() + 1e-6)

    return X.astype(np.float32, copy=False)

def transform_pair(mouse_pair, body_parts_tracked, fps):
    avail_A = mouse_pair['A'].columns.get_level_values(0)
    avail_B = mouse_pair['B'].columns.get_level_values(0)
    X = pd.DataFrame({
        f"12+{p1}+{p2}": np.square(mouse_pair['A'][p1] - mouse_pair['B'][p2]).sum(axis=1, skipna=False)
        for p1, p2 in itertools.product(body_parts_tracked, repeat=2)
        if p1 in avail_A and p2 in avail_B
    })
    X = X.reindex(columns=[f"12+{p1}+{p2}" for p1, p2 in itertools.product(body_parts_tracked, repeat=2)], copy=False)

    if ('A', 'ear_left') in mouse_pair.columns and ('B', 'ear_left') in mouse_pair.columns:
        lag = _scale(10, fps)
        shA = mouse_pair['A']['ear_left'].shift(lag)
        shB = mouse_pair['B']['ear_left'].shift(lag)
        speeds = pd.DataFrame({
            'sp_A': np.square(mouse_pair['A']['ear_left'] - shA).sum(axis=1, skipna=False),
            'sp_AB': np.square(mouse_pair['A']['ear_left'] - shB).sum(axis=1, skipna=False),
            'sp_B': np.square(mouse_pair['B']['ear_left'] - shB).sum(axis=1, skipna=False),
        })
        X = pd.concat([X, speeds], axis=1)

    if 'nose+tail_base' in X.columns and 'ear_left+ear_right' in X.columns:
        X['elong'] = X['nose+tail_base'] / (X['ear_left+ear_right'] + 1e-6)

    if all(p in avail_A for p in ['nose', 'tail_base']) and all(p in avail_B for p in ['nose', 'tail_base']):
        dir_A = mouse_pair['A']['nose'] - mouse_pair['A']['tail_base']
        dir_B = mouse_pair['B']['nose'] - mouse_pair['B']['tail_base']
        X['rel_ori'] = (dir_A['x'] * dir_B['x'] + dir_A['y'] * dir_B['y']) / (
            np.sqrt(dir_A['x']**2 + dir_A['y']**2) * np.sqrt(dir_B['x']**2 + dir_B['y']**2) + 1e-6)

    if all(p in avail_A for p in ['nose']) and all(p in avail_B for p in ['nose']):
        cur = np.square(mouse_pair['A']['nose'] - mouse_pair['B']['nose']).sum(axis=1, skipna=False)
        lag = _scale(10, fps)
        shA_n = mouse_pair['A']['nose'].shift(lag)
        shB_n = mouse_pair['B']['nose'].shift(lag)
        past = np.square(shA_n - shB_n).sum(axis=1, skipna=False)
        X['appr'] = cur - past

    if 'body_center' in avail_A and 'body_center' in avail_B:
        cd = np.sqrt((mouse_pair['A']['body_center']['x'] - mouse_pair['B']['body_center']['x'])**2 + (mouse_pair['A']['body_center']['y'] - mouse_pair['B']['body_center']['y'])**2)
        X['v_cls'] = (cd < 5.0).astype(float)
        X['cls']   = ((cd >= 5.0) & (cd < 15.0)).astype(float)
        X['med']   = ((cd >= 15.0) & (cd < 30.0)).astype(float)
        X['far']   = (cd >= 30.0).astype(float)

        cd_full = np.square(mouse_pair['A']['body_center'] - mouse_pair['B']['body_center']).sum(axis=1, skipna=False)
        for w in [5, 15, 30, 60]:
            ws = _scale(w, fps)
            roll = dict(min_periods=1, center=True)
            X[f'd_m{w}']  = cd_full.rolling(ws, **roll).mean()
            X[f'd_s{w}']  = cd_full.rolling(ws, **roll).std()
            X[f'd_mn{w}'] = cd_full.rolling(ws, **roll).min()
            X[f'd_mx{w}'] = cd_full.rolling(ws, **roll).max()
            d_var = cd_full.rolling(ws, **roll).var()
            X[f'int{w}'] = 1 / (1 + d_var)
            Axd = mouse_pair['A']['body_center']['x'].diff(); Ayd = mouse_pair['A']['body_center']['y'].diff()
            Bxd = mouse_pair['B']['body_center']['x'].diff(); Byd = mouse_pair['B']['body_center']['y'].diff()
            coord = Axd * Bxd + Ayd * Byd
            X[f'co_m{w}'] = coord.rolling(ws, **roll).mean()
            X[f'co_s{w}'] = coord.rolling(ws, **roll).std()

    if 'nose' in avail_A and 'nose' in avail_B:
        nn = np.sqrt((mouse_pair['A']['nose']['x'] - mouse_pair['B']['nose']['x'])**2 + (mouse_pair['A']['nose']['y'] - mouse_pair['B']['nose']['y'])**2)
        for lag in [10, 20, 40]:
            l = _scale(lag, fps)
            X[f'nn_lg{lag}']  = nn.shift(l)
            X[f'nn_ch{lag}']  = nn - nn.shift(l)
            is_cl = (nn < 10.0).astype(float)
            X[f'cl_ps{lag}']  = is_cl.rolling(l, min_periods=1).mean()

    if 'body_center' in avail_A and 'body_center' in avail_B:
        Avx = mouse_pair['A']['body_center']['x'].diff(); Avy = mouse_pair['A']['body_center']['y'].diff()
        Bvx = mouse_pair['B']['body_center']['x'].diff(); Bvy = mouse_pair['B']['body_center']['y'].diff()
        val = (Avx * Bvx + Avy * Bvy) / (np.sqrt(Avx**2 + Avy**2) * np.sqrt(Bvx**2 + Bvy**2) + 1e-6)
        for off in [-20, -10, 0, 10, 20]:
            o = _scale_signed(off, fps)
            X[f'va_{off}'] = val.shift(-o)
        w = _scale(30, fps)
        X['int_con'] = cd_full.rolling(w, min_periods=1, center=True).std() / (cd_full.rolling(w, min_periods=1, center=True).mean() + 1e-6)
        X = add_interaction_features(X, mouse_pair, avail_A, avail_B, fps)

    return X.astype(np.float32, copy=False)

# ============================================================
# Parallel Data Processing
# ============================================================

drop_body_parts = ['headpiece_bottombackleft', 'headpiece_bottombackright', 'headpiece_bottomfrontleft', 'headpiece_bottomfrontright', 
                   'headpiece_topbackleft', 'headpiece_topbackright', 'headpiece_topfrontleft', 'headpiece_topfrontright',                  
                   'spine_1', 'spine_2', 'tail_middle_1', 'tail_middle_2', 'tail_midpoint']

def _to_num(x):
    if isinstance(x, (int, np.integer)): return int(x)
    m = re.search(r'(\d+)$', str(x))
    return int(m.group(1)) if m else None

def process_video_wrapper(row, traintest, body_parts_tracked, traintest_directory, drop_body_parts):
    # Re-implementing generate_mouse_data logic for a single row
    import warnings
    warnings.filterwarnings('ignore')
    np.seterr(all='ignore') # Suppress numpy warnings in worker process

    lab_id   = row.lab_id
    video_id = row.video_id
    fps      = float(row.frames_per_second)
    n_mice   = int(row.n_mice)
    arena_w  = float(row.get('arena_width_cm', np.nan))
    arena_h  = float(row.get('arena_height_cm', np.nan))
    sleeping = bool(getattr(row, 'sleeping', False))
    arena_shape = row.get('arena_shape', 'rectangular')

    if not isinstance(row.behaviors_labeled, str):
        return []

    path = f"{traintest_directory}/{lab_id}/{video_id}.parquet"
    vid = pd.read_parquet(path)
    if len(np.unique(vid.bodypart)) > 5:
        vid = vid.query("~ bodypart.isin(@drop_body_parts)")
    pvid = vid.pivot(columns=['mouse_id','bodypart'], index='video_frame', values=['x','y'])
    del vid
    pvid = pvid.reorder_levels([1,2,0], axis=1).T.sort_index().T
    pvid = (pvid / float(row.pix_per_cm_approx)).astype('float32', copy=False)

    avail = list(pvid.columns.get_level_values('mouse_id').unique())
    avail_set = set(avail) | set(map(str, avail)) | {f"mouse{_to_num(a)}" for a in avail if _to_num(a) is not None}

    def _resolve(agent_str):
        m = re.search(r'(\d+)$', str(agent_str))
        cand = [agent_str]
        if m:
            n = int(m.group(1))
            cand = [n, n-1, str(n), f"mouse{n}", agent_str]
        for c in cand:
            if c in avail_set:
                if c in set(avail): return c
                for a in avail:
                    if str(a) == str(c) or f"mouse{_to_num(a)}" == str(c):
                        return a
        return None

    vb = json.loads(row.behaviors_labeled)
    vb = sorted(list({b.replace("'", "") for b in vb}))
    vb = pd.DataFrame([b.split(',') for b in vb], columns=['agent','target','action'])
    vb['agent']  = vb['agent'].astype(str)
    vb['target'] = vb['target'].astype(str)
    vb['action'] = vb['action'].astype(str).str.lower()

    annot = None
    if traintest == 'train':
        try:
            annot_path = path.replace('train_tracking', 'train_annotation')
            annot = pd.read_parquet(annot_path)
        except FileNotFoundError:
            pass

    def _mk_meta(index, agent_id, target_id):
        m = pd.DataFrame({
            'lab_id':        lab_id,
            'video_id':      video_id,
            'agent_id':      agent_id,
            'target_id':     target_id,
            'video_frame':   index.astype('int32', copy=False),
            'frames_per_second': np.float32(fps),
            'sleeping':      sleeping,
            'arena_shape':   arena_shape,
            'arena_width_cm': np.float32(arena_w),
            'arena_height_cm': np.float32(arena_h),
            'n_mice':        np.int8(n_mice),
        })
        for c in ('lab_id','video_id','agent_id','target_id','arena_shape'):
            m[c] = m[c].astype('category')
        return m

    results = []

    # Single
    vb_single = vb.query("target == 'self'")
    for agent_str in pd.unique(vb_single['agent']):
        col_lab = _resolve(agent_str)
        if col_lab is None: continue
        actions = sorted(vb_single.loc[vb_single['agent'].eq(agent_str), 'action'].unique().tolist())
        if not actions: continue

        single = pvid.loc[:, col_lab]
        meta_df = _mk_meta(single.index, agent_str, 'self')
        
        # Transform immediately to save memory
        X = transform_single(single, body_parts_tracked, fps)

        if traintest == 'train' and annot is not None:
            a_num = _to_num(col_lab)
            y = pd.DataFrame(False, index=single.index.astype('int32', copy=False), columns=actions)
            a_sub = annot.query("(agent_id == @a_num) & (target_id == @a_num)")
            for i in range(len(a_sub)):
                ar = a_sub.iloc[i]
                a = str(ar.action).lower()
                if a in y.columns:
                    y.loc[int(ar['start_frame']):int(ar['stop_frame']), a] = True
            results.append(('single', X, meta_df, y))
        else:
            results.append(('single', X, meta_df, actions))

    # Pair
    vb_pair = vb.query("target != 'self'")
    if len(vb_pair) > 0:
        allowed_pairs = set(map(tuple, vb_pair[['agent','target']].itertuples(index=False, name=None)))
        for agent_num, target_num in itertools.permutations(np.unique(pvid.columns.get_level_values('mouse_id')), 2):
            agent_str = f"mouse{_to_num(agent_num)}"
            target_str = f"mouse{_to_num(target_num)}"
            if (agent_str, target_str) not in allowed_pairs: continue

            a_col = _resolve(agent_str)
            b_col = _resolve(target_str)
            if a_col is None or b_col is None: continue

            actions = sorted(vb_pair.query("(agent == @agent_str) & (target == @target_str)")['action'].unique().tolist())
            if not actions: continue

            pair_xy = pd.concat([pvid[a_col], pvid[b_col]], axis=1, keys=['A','B'])
            meta_df = _mk_meta(pair_xy.index, agent_str, target_str)
            
            # Transform immediately
            X = transform_pair(pair_xy, body_parts_tracked, fps)

            if traintest == 'train' and annot is not None:
                a_num = _to_num(a_col); b_num = _to_num(b_col)
                y = pd.DataFrame(False, index=pair_xy.index.astype('int32', copy=False), columns=actions)
                a_sub = annot.query("(agent_id == @a_num) & (target_id == @b_num)")
                for i in range(len(a_sub)):
                    ar = a_sub.iloc[i]
                    a = str(ar.action).lower()
                    if a in y.columns:
                        y.loc[int(ar['start_frame']):int(ar['stop_frame']), a] = True
                results.append(('pair', X, meta_df, y))
            else:
                results.append(('pair', X, meta_df, actions))
    
    return results

# ============================================================
# Main Execution Loop
# ============================================================

def predict_multiclass_adaptive(pred, meta, action_thresholds=defaultdict(lambda: 0.27)):
    # Temporal smoothing (Mean Filter Window 5)
    pred_smoothed = pred.rolling(window=5, min_periods=1, center=True).mean()
    
    ama = np.argmax(pred_smoothed.values, axis=1)
    max_probs = pred_smoothed.max(axis=1)
    
    threshold_mask = np.zeros(len(pred_smoothed), dtype=bool)
    for i, action in enumerate(pred_smoothed.columns):
        action_mask = (ama == i)
        threshold = action_thresholds.get(action, 0.27)
        threshold_mask |= (action_mask & (max_probs >= threshold))
    
    ama = np.where(threshold_mask, ama, -1)
    ama = pd.Series(ama, index=meta.video_frame)
    
    changes_mask = (ama != ama.shift(1)).values
    ama_changes = ama[changes_mask]
    meta_changes = meta[changes_mask]
    mask = ama_changes.values >= 0
    mask[-1] = False
    
    submission_part = pd.DataFrame({
        'video_id': meta_changes['video_id'][mask].values,
        'agent_id': meta_changes['agent_id'][mask].values,
        'target_id': meta_changes['target_id'][mask].values,
        'action': pred.columns[ama_changes[mask].values],
        'start_frame': ama_changes.index[mask],
        'stop_frame': ama_changes.index[1:][mask[:-1]]
    })
    
    # Fix stop frames for video boundaries
    stop_video_id = meta_changes['video_id'][1:][mask[:-1]].values
    stop_agent_id = meta_changes['agent_id'][1:][mask[:-1]].values
    stop_target_id = meta_changes['target_id'][1:][mask[:-1]].values
    
    # Vectorized fix for boundaries? Iteration is safer for now
    for i in range(len(submission_part)):
        video_id = submission_part.video_id.iloc[i]
        agent_id = submission_part.agent_id.iloc[i]
        target_id = submission_part.target_id.iloc[i]
        if i < len(stop_video_id):
            if stop_video_id[i] != video_id or stop_agent_id[i] != agent_id or stop_target_id[i] != target_id:
                new_stop_frame = meta.query("(video_id == @video_id)").video_frame.max() + 1
                submission_part.iat[i, submission_part.columns.get_loc('stop_frame')] = new_stop_frame
        else:
            new_stop_frame = meta.query("(video_id == @video_id)").video_frame.max() + 1
            submission_part.iat[i, submission_part.columns.get_loc('stop_frame')] = new_stop_frame
            
    duration = submission_part.stop_frame - submission_part.start_frame
    submission_part = submission_part[duration >= 3].reset_index(drop=True)
    return submission_part

def robustify(submission, dataset, traintest, traintest_directory=None):
    if traintest_directory is None:
        traintest_directory = INPUT_DIR / f"{traintest}_tracking"

    submission = submission[submission.start_frame < submission.stop_frame]

    group_list = []
    for _, group in submission.groupby(['video_id', 'agent_id', 'target_id']):
        group = group.sort_values('start_frame')
        mask = np.ones(len(group), dtype=bool)
        last_stop = 0
        for i, (_, row) in enumerate(group.iterrows()):
            if row['start_frame'] < last_stop:
                mask[i] = False
            else:
                last_stop = row['stop_frame']
        group_list.append(group[mask])
    submission = pd.concat(group_list) if group_list else submission
    return submission

# Load Data
print("Loading Metadata...")
train = pd.read_csv(INPUT_DIR / 'train.csv')
train = train.loc[~(train['lab_id'].astype(str).str.contains('MABe22', na=False) &
                    train['mouse1_condition'].astype(str).str.lower().eq('lights on'))].copy()
train['n_mice'] = 4 - train[['mouse1_strain', 'mouse2_strain', 'mouse3_strain', 'mouse4_strain']].isna().sum(axis=1)

test = pd.read_csv(INPUT_DIR / 'test.csv')
test['sleeping'] = (test['lab_id'].astype(str).str.contains('MABe22', na=False) &
                    test['mouse1_condition'].astype(str).str.lower().eq('lights on'))
test['n_mice'] = 4 - test[['mouse1_strain','mouse2_strain','mouse3_strain','mouse4_strain']].isna().sum(axis=1)

body_parts_tracked_list = list(np.unique(train.body_parts_tracked))
submission_list = []

n_jobs = max(1, multiprocessing.cpu_count() - 1)
print(f"Using {n_jobs} jobs for parallel processing")

for section in range(len(body_parts_tracked_list)):
    body_parts_tracked_str = body_parts_tracked_list[section]
    body_parts_tracked = json.loads(body_parts_tracked_str)
    print(f"\n=== Section {section}: {len(body_parts_tracked)} body parts ===")
    if len(body_parts_tracked) > 5:
        body_parts_tracked = [b for b in body_parts_tracked if b not in drop_body_parts]

    # --- 1. TEST PHASE (Feature Generation) ---
    # Generate test features first to allow interleaved inference (saving memory)
    print("Generating Test Features...")
    test_subset = test[test.body_parts_tracked == body_parts_tracked_str]
    test_rows = [row for _, row in test_subset.iterrows()]
    
    test_results = Parallel(n_jobs=n_jobs)(
        delayed(process_video_wrapper)(row, 'test', body_parts_tracked, str(TEST_TRACKING_DIR), drop_body_parts) 
        for row in tqdm(test_rows)
    )
    
    # Organize Test Data by switch
    test_data_by_switch = {'single': [], 'pair': []}
    for sublist in test_results:
        for item in sublist:
            # item: (switch, X, meta, actions)
            if item[0] in test_data_by_switch:
                test_data_by_switch[item[0]].append({
                    'X': item[1],
                    'meta': item[2],
                    'actions': set(item[3]),
                    'preds': {} # To store predictions: action -> prob_array
                })
    del test_results; gc.collect()

    # --- 2. TRAIN PHASE (Feature Generation) ---
    print("Generating Training Features...")
    train_subset = train[train.body_parts_tracked == body_parts_tracked_str]
    train_rows = [row for _, row in train_subset.iterrows()]
    
    train_results = Parallel(n_jobs=n_jobs)(
        delayed(process_video_wrapper)(row, 'train', body_parts_tracked, str(TRAIN_TRACKING_DIR), drop_body_parts) 
        for row in tqdm(train_rows)
    )
    
    # Flatten and separate immediately to save memory
    single_train = []
    pair_train = []
    for sublist in train_results:
        for item in sublist:
            # item: (switch, X, meta, y)
            if isinstance(item[3], pd.DataFrame): # Filter valid
                if item[0] == 'single':
                    single_train.append(item)
                elif item[0] == 'pair':
                    pair_train.append(item)
    del train_results; gc.collect()
    
    # --- 3. TRAINING & INFERENCE LOOP ---
    for switch, data_list in [('single', single_train), ('pair', pair_train)]:
        if not data_list: continue
        print(f"Processing {switch} models...")
        
        # Prepare Train Data
        # We drop meta_tr to save RAM as it's not used for training
        X_tr = pd.concat([x[1] for x in data_list], axis=0, ignore_index=True)
        label_tr = pd.concat([x[3] for x in data_list], axis=0, ignore_index=True)
        del data_list; gc.collect()
        
        # Define Ensemble
        n_samples = 1_500_000
        models_def = []
        models_def.append(make_pipeline(StratifiedSubsetClassifier(_make_lgbm(n_estimators=225, learning_rate=0.07, min_child_samples=40, num_leaves=31, subsample=0.8, colsample_bytree=0.8, verbose=-1), n_samples)))
        models_def.append(make_pipeline(StratifiedSubsetClassifier(_make_lgbm(n_estimators=150, learning_rate=0.1, min_child_samples=20, num_leaves=63, max_depth=8, subsample=0.7, colsample_bytree=0.9, reg_alpha=0.1, reg_lambda=0.1, verbose=-1), (n_samples and int(n_samples/1.25)))))
        models_def.append(make_pipeline(StratifiedSubsetClassifier(_make_lgbm(n_estimators=100, learning_rate=0.05, min_child_samples=30, num_leaves=127, max_depth=10, subsample=0.75, verbose=-1), (n_samples and int(n_samples/1.66)))))
        models_def.append(make_pipeline(StratifiedSubsetClassifier(_make_xgb(n_estimators=180, learning_rate=0.08, max_depth=6, min_child_weight=8 if USE_GPU else 5, gamma=1.0 if USE_GPU else 0., subsample=0.8, colsample_bytree=0.8, verbosity=0), n_samples and int(n_samples/1.2))))
        models_def.append(make_pipeline(StratifiedSubsetClassifier(_make_cb(iterations=120, learning_rate=0.1, depth=6, verbose=False, allow_writing_files=False), n_samples)))
        
        if USE_GPU:
             models_def.append(make_pipeline(StratifiedSubsetClassifierWEval(xgb.XGBClassifier(random_state=SEED, tree_method="gpu_hist", n_estimators=2000, learning_rate=0.05, max_leaves=255, max_depth=0, min_child_weight=10, subsample=0.90, colsample_bytree=1.00, verbosity=0), n_samples and int(n_samples/2.))))
             models_def.append(make_pipeline(StratifiedSubsetClassifierWEval(xgb.XGBClassifier(random_state=SEED, tree_method="gpu_hist", n_estimators=1400, learning_rate=0.06, max_depth=7, min_child_weight=12, subsample=0.70, colsample_bytree=0.80, verbosity=0), n_samples and int(n_samples/1.5))))
             models_def.append(make_pipeline(StratifiedSubsetClassifierWEval(CatBoostClassifier(random_seed=SEED, task_type="GPU", devices="0", iterations=4000, learning_rate=0.03, depth=8, l2_leaf_reg=6.0, verbose=False, allow_writing_files=False), n_samples and int(n_samples/2.0))))

        for action in label_tr.columns:
            action_mask = ~label_tr[action].isna().values
            y_action = label_tr[action][action_mask].values.astype(int)
            X_action = X_tr[action_mask]
            
            trained_action_models = []
            for m in models_def:
                m_clone = clone(m)
                try:
                    m_clone.fit(X_action, y_action)
                    trained_action_models.append(m_clone)
                except Exception as e:
                    print(f"Failed to train model for {action}: {e}")
            
            print(f"  Trained {len(trained_action_models)} models for {action}")
            
            # Predict Immediately on Test Data
            test_items = test_data_by_switch.get(switch, [])
            for item in test_items:
                if action in item['actions']:
                    probs = []
                    for mdl in trained_action_models:
                        try:
                            probs.append(mdl.predict_proba(item['X'])[:, 1])
                        except: pass
                    if probs:
                        item['preds'][action] = np.mean(probs, axis=0)
            
            # Delete Models to free GPU/RAM immediately
            del trained_action_models, m_clone; gc.collect()
            
        del X_tr, label_tr; gc.collect()
        
        # Process Predictions for this switch
        test_items = test_data_by_switch.get(switch, [])
        for item in test_items:
            if item['preds']:
                pred_df = pd.DataFrame(item['preds'], index=item['meta'].video_frame)
                submission_list.append(predict_multiclass_adaptive(pred_df, item['meta']))
        
        # Clear test items for this switch
        test_data_by_switch[switch] = []
        gc.collect()

    del single_train, pair_train, test_data_by_switch; gc.collect()

# ============================================================
# Final Submission
# ============================================================
if len(submission_list) > 0:
    submission = pd.concat(submission_list, ignore_index=True)
else:
    # Dummy submission if empty
    submission = pd.DataFrame({
        'video_id': [438887472], 'agent_id': ['mouse1'], 'target_id': ['self'],
        'action': ['rear'], 'start_frame': [278], 'stop_frame': [500]
    })

submission_robust = robustify(submission, test, 'test')
submission_robust.index.name = 'row_id'
submission_robust.to_csv('submission.csv')
print(f"\nSubmission created: {len(submission_robust)} predictions")
!head -n 10 submission.csv

Using GPU? True
Loading Metadata...
Using 3 jobs for parallel processing

=== Section 0: 12 body parts ===
Generating Test Features...


0it [00:00, ?it/s]

Generating Training Features...


  0%|          | 0/7926 [00:00<?, ?it/s]


=== Section 1: 18 body parts ===
Generating Test Features...


...

Processing single models...
  Trained 8 models for rear
  Trained 8 models for selfgroom
  Trained 8 models for genitalgroom
  Trained 8 models for dig
Processing pair models...
  Trained 8 models for approach
  Trained 8 models for attack
  Trained 8 models for disengage
  Trained 8 models for mount
  Trained 8 models for sniff
  Trained 8 models for sniffgenital
  Trained 8 models for dominancemount
  Trained 8 models for sniffbody
  Trained 8 models for sniffface
  Trained 8 models for attemptmount
  Trained 8 models for intromit
  Trained 8 models for chase
  Trained 8 models for escape
  Trained 8 models for reciprocalsniff
  Trained 8 models for allogroom
  Trained 8 models for ejaculate
  Trained 8 models for dominancegroom

=== Section 9: 5 body parts ===
Generating Test Features...


0it [00:00, ?it/s]

Generating Training Features...


  0%|          | 0/24 [00:00<?, ?it/s]

Processing single models...
  Trained 8 models for freeze
  Trained 8 models for rear
Processing pair models...
  Trained 8 models for approach
  Trained 8 models for attack
  Trained 8 models for defend
  Trained 8 models for escape
  Trained 8 models for sniff

Submission created: 447 predictions
row_id,video_id,agent_id,target_id,action,start_frame,stop_frame
247,438887472,mouse1,mouse3,approach,2286,2292
248,438887472,mouse1,mouse4,submit,1424,1431
249,438887472,mouse1,mouse4,attack,1455,1466
250,438887472,mouse1,mouse4,attack,1467,1483
251,438887472,mouse1,mouse4,attack,1509,1525
252,438887472,mouse1,mouse4,submit,2382,2389
253,438887472,mouse1,mouse4,avoid,2562,2590
0,438887472,mouse1,self,rear,2128,2135
1,438887472,mouse1,self,rear,2178,2182
